In [1]:
# Load df_artists and df_songs from project CSVs
import pandas as pd

base = '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/'

df_artists = pd.read_csv(base + 'df_artists.csv')
df_songs   = pd.read_csv(base + 'df_songs.csv')

print(f'df_artists: {df_artists.shape}')
print(f'df_songs:   {df_songs.shape}')

df_artists: (13609, 54)
df_songs:   (38383, 43)


In [4]:
# Diagnostic: flag each issue type in df_artists to understand scope
import re

flags = {
    'starts_with_ampersand':  df_artists['name'].str.match(r'^&'),
    'collab_x':               df_artists['name'].str.contains(r'\bx\b', regex=True),
    'collab_slash':           df_artists['name'].str.contains(r'/', regex=False),
    'collab_comma':           df_artists['name'].str.contains(r', ', regex=False),
    'collab_or':              df_artists['name'].str.contains(r'\bor\b', regex=True),
    'collab_presents':        df_artists['name'].str.contains(r'\bpresents\b', regex=True),
    'collab_feat':            df_artists['name'].str.contains(r'\bfeat\b|\bft\b|\bfeaturing\b', regex=True),
    'parenthetical':          df_artists['name'].str.contains(r'\(', regex=True),
    'curly_quotes':           df_artists['name'].str.contains(r'[""'']', regex=True),
    'dollar_sign':            df_artists['name'].str.contains(r'\$', regex=False),
    'orchestra':              df_artists['name'].str.contains(r'\borchestra\b|\bband\b|\bhis \b|\bher \b', regex=True),
}

for label, mask in flags.items():
    n = mask.sum()
    print(f'{label:<30} {n:>5} rows')
    if n <= 5:
        print('  ', df_artists.loc[mask, 'name'].tolist())


starts_with_ampersand              6 rows
collab_x                          80 rows
collab_slash                     185 rows
collab_comma                     118 rows
collab_or                         26 rows
collab_presents                   37 rows
collab_feat                       66 rows
parenthetical                    131 rows
curly_quotes                      39 rows
dollar_sign                        0 rows
   []
orchestra                        366 rows


In [6]:
# Strip all quote characters for duplicate comparison
def normalize_for_dedup(s):
    s = re.sub(r'[""«»""\'\'‛`]', '', s)  # remove all quote variants
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_artists['name_dedup'] = df_artists['name'].apply(normalize_for_dedup)

dupes = df_artists[df_artists.duplicated('name_dedup', keep=False)].sort_values('name_dedup')
print(f'Duplicate pairs after stripping all quotes: {len(dupes)}')
print(dupes[['name', 'name_dedup']].to_string())


Duplicate pairs after stripping all quotes: 32
                             name                 name_dedup
1215       billy "crash" craddock       billy crash craddock
1217       billy 'crash' craddock       billy crash craddock
1462        bobby "boris" pickett        bobby boris pickett
1463        bobby 'boris' pickett        bobby boris pickett
2789     damian "jr. gong" marley     damian jr. gong marley
2790     damian 'jr. gong' marley     damian jr. gong marley
5782             jimmy "bo" horne             jimmy bo horne
5783             jimmy 'bo' horne             jimmy bo horne
6473                       keb mo                     keb mo
6474                     keb' mo'                     keb mo
6609         kierra 'kiki' sheard         kierra kiki sheard
6608         kierra "kiki" sheard         kierra kiki sheard
7132                  lil' boosie                 lil boosie
7084                   lil boosie                 lil boosie
7147                    lil' troy     

In [12]:
# Revised pick_best — no reference to name_dedup inside the function
KEEP_OVERRIDES = {"me'shell ndegeocello", "sniff 'n' the tears"}

def pick_best(group):
    for preferred in KEEP_OVERRIDES:
        match = group[group['name'] == preferred]
        if len(match):
            return match.index[0]
    return group['total_charting_songs'].idxmax()

dup_rows  = df_artists[df_artists.duplicated('name_dedup', keep=False)]
keep_idxs = dup_rows.groupby('name_dedup', group_keys=False).apply(pick_best).values
drop_idxs = dup_rows.index.difference(keep_idxs)

print(f'Dropping {len(drop_idxs)} rows. Keeping:')
print(df_artists.loc[keep_idxs, ['name', 'total_charting_songs']].to_string())


Dropping 16 rows. Keeping:
                            name  total_charting_songs
1215      billy "crash" craddock                     3
1462       bobby "boris" pickett                     2
2789    damian "jr. gong" marley                     1
5782            jimmy "bo" horne                     1
6473                      keb mo                     0
6608        kierra "kiki" sheard                     1
7084                  lil boosie                     2
7147                   lil' troy                     1
7122                   lil wayne                   180
7890        me'shell ndegeocello                     1
8402               nat king cole                    30
9735    richard "dimples" fields                     1
10719        sniff 'n' the tears                     1
11915  the edwin hawkins singers                     1
12800        the sopwith "camel"                     2
4            "weird al" yankovic                    11


/var/folders/1v/h2vyfq9j45n9_tmm_gg2s8tm0000gn/T/ipykernel_29612/2905915789.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keep_idxs = dup_rows.groupby('name_dedup', group_keys=False).apply(pick_best).values


In [13]:
# Commit deduplication
df_artists = df_artists.drop(index=drop_idxs).drop(columns=['name_dedup', 'name_normalized'])
print(f'df_artists after dedup: {df_artists.shape}')


df_artists after dedup: (14208, 54)


In [15]:
# High-confidence drops with exception for known false positives
KEEP_EXCEPTIONS = {'little feat'}

drop_mask = (
    (
        df_artists['name'].str.match(r'^&')
      | df_artists['name'].str.contains(r'\bfeat\b|\bft\.?\b|\bfeaturing\b', regex=True)
      | df_artists['name'].str.contains(r' x ', regex=False)
    )
    & ~df_artists['name'].isin(KEEP_EXCEPTIONS)
)

print(f'Rows to drop: {drop_mask.sum()}')
print(df_artists.loc[drop_mask, 'name'].to_string())


Rows to drop: 132
7                                       $uicideboy$ x germ
8                                         & cardi b or nas
9                                            & jack harlow
10                                           & johnny cash
11                                              & mystikal
12                                               & sam dew
13                            & youngboy never broke again
40                           2 chainz x gucci mane x quavo
112                                50 cent feat. mobb deep
647                                        arcangel x sech
661                           ariana grande feat. doja cat
788                                   aventura x bad bunny
853                      babyface (featuring toni braxton)
1127                      big daddy kane feat. spinderella
1344                       blackstreet (featuring dr. dre)
1912                calle 24 x chino pacas x fuerza regida
2134                      changing fac

In [16]:
df_artists = df_artists[~drop_mask].copy()
print(f'df_artists after dropping collabs/features: {df_artists.shape}')


df_artists after dropping collabs/features: (14076, 54)


In [17]:
# Preview 'presents' entries
presents_mask = df_artists['name'].str.contains(r'\bpresents\b', regex=True)
print(f'{presents_mask.sum()} rows:')
print(df_artists.loc[presents_mask, 'name'].to_string())


37 rows:
169         a.b. quintanilla iii presents kumbia all starz
170             a.b. quintanilla iii presents kumbia kings
599                            ant banks presents t.w.d.y.
1283     bishop eddie l. long presents new birth total ...
1285     bishop noel jones presents the city of refuge ...
1287            bishop paul s. morton presents the fgbcfmc
1722              broken social scene presents: kevin drew
1876                c+c music factory presents zelma davis
1923                            cam'ron presents dukedagod
1924                        cam'ron presents the diplomats
2459                           clipse presents: re-up gang
2799                         damizza presents shade sheist
3282                          diplo presents thomas wesley
3283                      diplomats presents freekey zekey
3284                         diplomats presents: hell rell
3321     dj drama presents: wyclef jean aka toussaint s...
3332                           dj khaled presen

In [18]:
# Drop 'presents' entries
df_artists = df_artists[~presents_mask].copy()
print(f'df_artists after dropping presents: {df_artists.shape}')


df_artists after dropping presents: (14039, 54)


In [19]:
# Preview comma-separated entries (likely multi-artist listings)
comma_mask = df_artists['name'].str.contains(r', ', regex=False)
print(f'{comma_mask.sum()} rows:')
print(df_artists.loc[comma_mask, 'name'].to_string())


117 rows:
51                          21 savage & tyler, the creator
52                     21 savage, travis scott & yak gotti
160                        a$ap rocky & tyler, the creator
278                                 akon, lil wayne & niia
536      andy nyman, courtney-mae briggs, jeff goldblum...
662      ariana grande, ethan slater, marissa bode & cy...
704                         artists of then, now & forever
727                                 ashton, gardner & dyke
838                 baaba maal, the very best & beatenberg
963      bas, jid, guapdad 4000, reese laflare, jace, m...
976                   battle, von stade, marsalis (previn)
1053               benedictines of mary, queen of apostles
1174                         bilal, anna wise & snoop dogg
1175                         bilal, anna wise & thundercat
1277                        birdman, jay sean, & lil wayne
1305                               black country, new road
1308                   black eyed peas, ozuna 

In [20]:
# Keep comma entries that are legitimate single-artist names; drop multi-artist collabs
COMMA_KEEP = {
    # Jr./Sr./I suffixes
    'bobby pedrick, jr.', 'carl dobkins, jr.', 'grover washington, jr.',
    'hank williams, jr.', 'harry connick, jr.', 'karl hammel, jr.',
    'oscar toney, jr.', 'sammy davis, jr.', 'rev. dr. martin luther king, jr.',
    'rev. martin luther king, jr.', 'landau eugene murphy, jr.',
    'robert downey, jr.', 'john wesley ryles, i',
    # Band names with commas
    'tyler, the creator', 'blood, sweat & tears', 'earth, wind & fire',
    'emerson, lake & palmer', 'emerson, lake & powell',
    'crosby, stills & nash', 'crosby, stills, nash & young',
    'peter, paul & mary', 'ray, goodman & brown',
    'hamilton, joe frank & dennison', 'hamilton, joe frank & reynolds',
    'dino, desi & billy', 'lipps, inc.', 'isley, jasper, isley',
    'ecstasy, passion & pain', 'faith, hope and charity',
    'dancer, prancer and nervous', 'west, bruce & laing',
    'black country, new road', 'mcguinn, clark & hillman',
    'phantom, rocker & slick', 'hagar, schon, aaronson, shrieve',
    'ashton, gardner & dyke', 'phillips, craig & dean',
    'invent, animate', 'drop dead, gorgeous', 'oh, sleeper',
    'woe, is me', 'whenever, if ever', 'chunk! no, captain chunk!',
    'benedictines of mary, queen of apostles',
    'hodges, james and smith', 'cotton, lloyd and christian',
    'three hanks: hank williams, sr., jr., iii',
}

comma_mask = df_artists['name'].str.contains(r', ', regex=False)
comma_drop = comma_mask & ~df_artists['name'].isin(COMMA_KEEP)

print(f'Dropping {comma_drop.sum()} rows, keeping {(comma_mask & df_artists["name"].isin(COMMA_KEEP)).sum()}')
print('\nDropping:')
print(df_artists.loc[comma_drop, 'name'].to_string())
print('\nKeeping:')
print(df_artists.loc[comma_mask & df_artists['name'].isin(COMMA_KEEP), 'name'].to_string())


Dropping 71 rows, keeping 46

Dropping:
51                          21 savage & tyler, the creator
52                     21 savage, travis scott & yak gotti
160                        a$ap rocky & tyler, the creator
278                                 akon, lil wayne & niia
536      andy nyman, courtney-mae briggs, jeff goldblum...
662      ariana grande, ethan slater, marissa bode & cy...
704                         artists of then, now & forever
838                 baaba maal, the very best & beatenberg
963      bas, jid, guapdad 4000, reese laflare, jace, m...
976                   battle, von stade, marsalis (previn)
1174                         bilal, anna wise & snoop dogg
1175                         bilal, anna wise & thundercat
1277                        birdman, jay sean, & lil wayne
1308                   black eyed peas, ozuna + j.rey soul
2026                  carreras, domingo, pavarotti (mehta)
2093              cedric gervais, wiz khalifa & mally mall
2278            

In [22]:
# Commit comma drops
df_artists = df_artists[~comma_drop].copy()
print(f'df_artists after dropping comma collabs: {df_artists.shape}')


df_artists after dropping comma collabs: (13968, 54)


In [23]:
# Preview 'or' entries
or_mask = df_artists['name'].str.contains(r'\bor\b', regex=True)
print(f'{or_mask.sum()} rows:')
print(df_artists.loc[or_mask, 'name'].to_string())


19 rows:
339                            alex band or chad kroeger
1230                    billy currington or mark mcgrath
3050                                       dead or alive
3369                                           do or die
3728                            el cata or dizzee rascal
4080                                     fight or flight
5161                          iamsu & skipper or 50 cent
5203                              iggy azalea or pitbull
6913                   lauren alaina or mackenzie porter
7123     lil wayne & french montana or too $hort or tyga
7126                                   lil wayne or t.i.
8553             nicki minaj or chimamanda ngozi adichie
9252                             plies or oj da juiceman
9277                              pooh shiesty or dababy
10499                      sheryl crow or allison moorer
11191                           swae lee or rae sremmurd
11300                                   tamia or ashanti
13142                 

In [24]:
# Drop 'or' collab entries, keep legitimate band names
OR_KEEP = {'dead or alive', 'do or die', 'fight or flight'}

or_mask = df_artists['name'].str.contains(r'\bor\b', regex=True)
or_drop = or_mask & ~df_artists['name'].isin(OR_KEEP)

print(f'Dropping {or_drop.sum()}, keeping {or_mask.sum() - or_drop.sum()}')
df_artists = df_artists[~or_drop].copy()
print(f'df_artists: {df_artists.shape}')


Dropping 16, keeping 3
df_artists: (13952, 54)


In [25]:
# Preview slash entries
slash_mask = df_artists['name'].str.contains('/', regex=False)
print(f'{slash_mask.sum()} rows:')
print(df_artists.loc[slash_mask, 'name'].to_string())


183 rows:
48                                                   20/20
59                       2cellos/london symphony orchestra
202                                                  ac/dc
264                          agnetha faltskog/peter cetera
280                                          akwid / jae-p
291               al dimeola/john mclaughlin/paco de lucia
307               alan jackson/george strait/jimmy buffett
337                                    alesso / katy perry
481                              ana barbara/jennifer pena
610               anthony kearns/ronan tynan/finbar wright
657                        aretha franklin/whitney houston
936                          barbra streisand/donna summer
1043                                 ben folds/nick hornby
1071             berlin philharmonic/mstislav rostropovich
1226                         billy cobham/george duke band
1434                                    bob dylan/the band
1439                               bob james/d

In [26]:
SLASH_KEEP = {
    # Band names with slash
    'ac/dc', '20/20', 'ebn/ozn', 'k/da', 'love/hate', 'm/a/r/r/s',
    'gza/genius', 'genius/gza', 'roni size/reprazent', 'zappa/mothers',
    # Artist + regular backing band
    'neil young/crazy horse', 'neil young / crazy horse',
    'neil young/international harvesters',
    'john lennon/plastic ono band', 'john & yoko/the plastic ono band',
    'john lennon / yoko ono',
    'jim morrison / the doors', 'john fogerty/the blue ridge rangers',
    'paul kantner/jefferson starship', 'rufus/chaka khan',
    'ozzy osbourne/randy rhoads', 'sugarloaf/jerry corbetta',
    'the tonight show band/doc severinsen', 'the raelets/ray charles orchestra',
    # Established long-running duos/groups
    'mojo nixon/skid roper', 'redman/method man', 'coverdale/page',
    'bob james/david sanborn', 'bob james/earl klugh',
    'stanley clarke/george duke', 'george benson/earl klugh',
    'david crosby/graham nash', 'dan fogelberg/tim weisberg',
    'courtney barnett / kurt vile', 'case/lang/veirs',
    'ian hunter/mick ronson', 'peabo bryson/roberta flack',
    'starlito / don trip', 'the benoit/freeman project',
    'the sanford/townsend band', 'the tarney/spencer band',
    'the clayton/hamilton jazz orchestra', 'hughes/thrall',
    'czarface / mf doom', 'al dimeola/john mclaughlin/paco de lucia',
    'keith jarrett/gary peacock/jack dejohnette',
    'julie driscoll/brian auger', 'bob dylan/the band',
    # Added
    'dee gees / foo fighters',
    'harry connick, jr./2006 broadway cast recording',
}

slash_mask = df_artists['name'].str.contains('/', regex=False)
slash_drop = slash_mask & ~df_artists['name'].isin(SLASH_KEEP)

print(f'Dropping {slash_drop.sum()}, keeping {(slash_mask & df_artists["name"].isin(SLASH_KEEP)).sum()}')
print('\nDropping:')
print(df_artists.loc[slash_drop, 'name'].to_string())
print('\nKeeping:')
print(df_artists.loc[slash_mask & df_artists['name'].isin(SLASH_KEEP), 'name'].to_string())


Dropping 134, keeping 49

Dropping:
59                       2cellos/london symphony orchestra
264                          agnetha faltskog/peter cetera
280                                          akwid / jae-p
307               alan jackson/george strait/jimmy buffett
337                                    alesso / katy perry
481                              ana barbara/jennifer pena
610               anthony kearns/ronan tynan/finbar wright
657                        aretha franklin/whitney houston
936                          barbra streisand/donna summer
1043                                 ben folds/nick hornby
1071             berlin philharmonic/mstislav rostropovich
1226                         billy cobham/george duke band
1511                           bobby rydell/chubby checker
1541                                boney james/rick braun
1582                     boris karloff / thurl ravenscroft
1646                branford marsalis quartet/t. blanchard
1724                

In [27]:
# Commit slash drops
df_artists = df_artists[~slash_drop].copy()
print(f'df_artists after slash drops: {df_artists.shape}')


df_artists after slash drops: (13818, 54)


In [28]:
# Preview parenthetical entries
paren_mask = df_artists['name'].str.contains(r'\(', regex=True)
print(f'{paren_mask.sum()} rows:')
print(df_artists.loc[paren_mask, 'name'].to_string())


91 rows:
0                                        !!! (chk chk chk)
18                                                   (+44)
19                                                (g)i-dle
20                                              (hed) p.e.
21                                      (hed) planet earth
22                             (the preacher) bobby womack
96                                     4*town (from disney
134                                      ? (question mark)
281                                al (he's the king) hirt
469                             amil (of major coinz) & ja
473                  amsterdam baroque orchestra (koopman)
568                                    anita cochran (duet
823                               b.c.g. (b.c. generation)
830                              b.m.u. (black men united)
899                               band (ohio untouchables)
1072                       berliner symphoniker (marturet)
1130                       big dee irwin (with 

In [30]:
# Drop parenthetical entries that are duet credits, conductor qualifiers, or role qualifiers
# Keep entries where parentheses are part of the artist name
PAREN_KEEP = {
    '!!! (chk chk chk)', '(+44)', '(g)i-dle', '(hed) p.e.',
    '(hed) planet earth', '? (question mark)', 'b.m.u. (black men united)',
    'f.l.y. (fast life yungstaz)', 'm.v.p. (most valuable playas)',
    'was (not was)', 'ernie (jim henson)', 'kermit (jim henson)',
    'ice cream man (master p)', 'pooh-man (mc pooh)', 'tyrese (aka black-ty)',
    'bri (briana babineaux)', 'sky (arista)', 'shirley (& company)',
    'b.c.g. (b.c. generation)', 'the valentinos (the lovers)',
    'the hippies (formerly the tams)', '4*town (from disney',
    'mount westmore (snoop dogg', 'silk sonic (bruno mars',
    'the swell season (glen hansard', 'the barbusters (joan jett',
    'the ruffin brothers (jimmy', 'i.a.p. co. (the italian asphalt',
    'mr. marcelo (from the ghetto)',
}

paren_mask = df_artists['name'].str.contains(r'\(', regex=True)
paren_drop = paren_mask & ~df_artists['name'].isin(PAREN_KEEP)

print(f'Dropping {paren_drop.sum()}, keeping {(paren_mask & df_artists["name"].isin(PAREN_KEEP)).sum()}')
print('\nDropping:')
print(df_artists.loc[paren_drop, 'name'].to_string())
print('\nKeeping:')
print(df_artists.loc[paren_mask & df_artists['name'].isin(PAREN_KEEP), 'name'].to_string())

df_artists = df_artists[~paren_drop].copy()
print(f'\ndf_artists after parenthetical drops: {df_artists.shape}')


Dropping 62, keeping 29

Dropping:
22                             (the preacher) bobby womack
281                                al (he's the king) hirt
469                             amil (of major coinz) & ja
473                  amsterdam baroque orchestra (koopman)
568                                    anita cochran (duet
899                               band (ohio untouchables)
1072                       berliner symphoniker (marturet)
1130                       big dee irwin (with little eva)
1815                            buddy guy (with g.e. smith
1929                      camerata antonio lucio (francis)
2027                   carreras-domingo-pavarotti (levine)
2281                    chris christian (with amy holland)
2358                   chubby checker (with dee dee sharp)
2814                                        dan hill (duet
3274                                        dion (di muci)
3389                                    dolly parton (duet
3681                 

In [32]:
# Rescue the singing nun from the original CSV
original = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv')
singing_nun = original[original['name'] == 'the singing nun (soeur sourire)'].copy()
singing_nun['name'] = 'the singing nun'
df_artists = pd.concat([df_artists, singing_nun], ignore_index=True)
print(f'df_artists after re-adding singing nun: {df_artists.shape}')
print(df_artists[df_artists['name'] == 'the singing nun'][['name', 'total_charting_songs', 'first_song_year']])


df_artists after re-adding singing nun: (13757, 54)
                  name  total_charting_songs  first_song_year
13756  the singing nun                     1           1963.0


In [33]:
# Preview orchestra/ensemble entries
orch_mask = df_artists['name'].str.contains(r'\borchestra\b|\bband\b|\bhis \b|\bher \b', regex=True)
print(f'{orch_mask.sum()} rows:')
print(df_artists.loc[orch_mask, 'name'].to_string())


338 rows:
136                                      a ragamuffin band
213                                               adc band
263                            al caiola and his orchestra
369                                        all sports band
381                                   allman brothers band
409              amanda palmer & the grand theft orchestra
634                       arkansas junior high school band
637                                       armada orchestra
652                           art mooney and his orchestra
752                                     average white band
785          b.b.king and the royal philharmonic orchestra
827                                      bad boy's da band
847                                      baja marimba band
853                                      ballas hough band
859                                                   band
860                                            band aid 30
861                                           

In [34]:
# Step 1: Drop artifact fragments and generic entries
orch_drop = df_artists['name'].isin({
    'his buckingham road quintet', 'his clowns', 'his cowboy outfit',
    'his electric eclectics', 'his friends', 'his guitar',
    'his johann strauss orchestra', 'his musette orchestra', 'his orch.',
    'his pianos', 'his talking steel guitar', 'his whispering organ sound',
    'orchestra', 'orchestra conducted by gordon jenkins',
})
df_artists = df_artists[~orch_drop].copy()
print(f'df_artists after artifact drops: {df_artists.shape}')


df_artists after artifact drops: (13743, 54)


In [35]:
# Step 2: Find duplicates caused by leading "the"
def strip_leading_the(s):
    return re.sub(r'^the ', '', s).strip()

df_artists['name_no_the'] = df_artists['name'].apply(strip_leading_the)

the_dupes = df_artists[df_artists.duplicated('name_no_the', keep=False)].sort_values('name_no_the')
print(f'Duplicate pairs with/without "the": {len(the_dupes)}')
print(the_dupes[['name', 'name_no_the', 'total_charting_songs']].to_string())


Duplicate pairs with/without "the": 260
                                       name                       name_no_the  total_charting_songs
97                             5 stairsteps                      5 stairsteps                     2
11067                      the 5 stairsteps                      5 stairsteps                     2
217                        addrisi brothers                  addrisi brothers                     6
11077                  the addrisi brothers                  addrisi brothers                     0
299                               alchemist                         alchemist                     0
11088                         the alchemist                         alchemist                     1
11094              the allman brothers band              allman brothers band                    10
381                    allman brothers band              allman brothers band                     0
11115                      the art of noise                 

In [36]:
# Keep the variant with more charting songs for each with/without-"the" pair, then commit
def pick_best_the(group):
    return group['total_charting_songs'].idxmax()

the_dup_rows = df_artists[df_artists.duplicated('name_no_the', keep=False)]
keep_idxs = the_dup_rows.groupby('name_no_the', group_keys=False).apply(
    pick_best_the, include_groups=False
).values
drop_idxs = the_dup_rows.index.difference(keep_idxs)

print(f'Dropping {len(drop_idxs)} rows')
df_artists = df_artists.drop(index=drop_idxs).drop(columns=['name_no_the'])
print(f'df_artists after "the" dedup: {df_artists.shape}')


Dropping 130 rows
df_artists after "the" dedup: (13613, 54)


In [38]:
# Merge Five Stairsteps variants — keep the one with most songs
stairsteps_drop = df_artists['name'].isin({'5 stairsteps', 'the stairsteps'})
df_artists = df_artists[~stairsteps_drop].copy()
print(f'df_artists after stairsteps fix: {df_artists.shape}')


df_artists after stairsteps fix: (13611, 54)


In [39]:
# Check which variant was kept for each pair
all_dup_names = {
    '5 stairsteps', 'the 5 stairsteps', 'addrisi brothers', 'the addrisi brothers',
    'alchemist', 'the alchemist', 'allman brothers band', 'the allman brothers band',
    'art of noise', 'the art of noise', 'atlanta rhythm section', 'the atlanta rhythm section',
    'baja marimba band', 'the baja marimba band', 'band', 'the band',
    'band of the black watch', 'the band of the black watch', 'bar-kays', 'the bar-kays',
    'beau brummels', 'the beau brummels', 'biddu orchestra', 'the biddu orchestra',
    'black eyed peas', 'the black eyed peas', 'brooklyn bridge', 'the brooklyn bridge',
    'brotherhood of man', 'the brotherhood of man', 'buggles', 'the buggles',
    'bus boys', 'the bus boys', 'captain & tennille', 'the captain & tennille',
    'casinos', 'the casinos', 'cathedrals', 'the cathedrals',
    'chambers brothers', 'the chambers brothers', 'charlie daniels band', 'the charlie daniels band',
    'chopper city boyz', 'the chopper city boyz', 'climax blues band', 'the climax blues band',
    'crystal mansion', 'the crystal mansion', 'defranco family', 'the defranco family',
    'dirty heads', 'the dirty heads', 'don harrison band', 'the don harrison band',
    'doobie brothers', 'the doobie brothers', 'dreamlovers', 'the dreamlovers',
    'dropkick murphys', 'the dropkick murphys', 'dukays', 'the dukays',
    'dynamic superiors', 'the dynamic superiors', 'electric light orchestra', 'the electric light orchestra',
    'fabulous poodles', 'the fabulous poodles', 'fantastic four', 'the fantastic four',
    'fireballs', 'the fireballs', 'fireflies', 'the fireflies',
    'first class', 'the first class', 'five', 'the five',
    'five stairsteps', 'the five stairsteps', 'flame', 'the flame',
    'flamingos', 'the flamingos', 'free movement', 'the free movement',
    'fugees', 'the fugees', 'fuzz', 'the fuzz',
    'game', 'the game', 'goo goo dolls', 'the goo goo dolls',
    'grateful dead', 'the grateful dead', 'hawks', 'the hawks',
    'heart', 'the heart', 'henry paul band', 'the henry paul band',
    'house of love', 'the house of love', 'illusion', 'the illusion',
    'incredible bongo band', 'the incredible bongo band', 'independents', 'the independents',
    'jackson 5', 'the jackson 5', 'jacksons', 'the jacksons',
    'james gang', 'the james gang', 'jimi hendrix experience', 'the jimi hendrix experience',
    'kalin twins', 'the kalin twins', 'kenny wayne shepherd band', 'the kenny wayne shepherd band',
    'kid laroi', 'the kid laroi', 'lemon pipers', 'the lemon pipers',
    'london philharmonic orchestra', 'the london philharmonic orchestra',
    'london symphony orchestra', 'the london symphony orchestra',
    'magic lanterns', 'the magic lanterns', 'mahavishnu orchestra', 'the mahavishnu orchestra',
    'mar-keys', 'the mar-keys', 'marmalade', 'the marmalade',
    'marshall tucker band', 'the marshall tucker band', 'memphis horns', 'the memphis horns',
    'meters', 'the meters', 'metropole orkest', 'the metropole orkest',
    'michael stanley band', 'the michael stanley band', 'mighty clouds of joy', 'the mighty clouds of joy',
    'moments', 'the moments', 'mormon tabernacle choir', 'the mormon tabernacle choir',
    'nails', 'the nails', 'neon philharmonic', 'the neon philharmonic',
    'new birth', 'the new birth', 'new broadway cast recording', 'the new broadway cast recording',
    'new colony six', 'the new colony six', 'nightcrawlers', 'the nightcrawlers',
    'nitty gritty dirt band', 'the nitty gritty dirt band', 'notorious b.i.g.', 'the notorious b.i.g.',
    'original broadway cast recording', 'the original broadway cast recording',
    'original cast recording', 'the original cast recording',
    'outlaws', 'the outlaws', 'outlawz', 'the outlawz',
    'paris sisters', 'the paris sisters', "people's choice", "the people's choice",
    'peppermint rainbow', 'the peppermint rainbow', 'plastic ono band', 'the plastic ono band',
    'prodigy', 'the prodigy', 'psychedelic furs', 'the psychedelic furs',
    'radiants', 'the radiants', 'raeletts', 'the raeletts',
    'ramones', 'the ramones', 'raspberries', 'the raspberries',
    'ray bryant combo', 'the ray bryant combo', 'ren', 'the ren',
    'robbin thompson band', 'the robbin thompson band', 'robert cray band', 'the robert cray band',
    'rolling stones', 'the rolling stones', 'rose', 'the rose',
    'rossington collins band', 'the rossington collins band',
    'royal philharmonic orchestra', 'the royal philharmonic orchestra',
    'salsoul orchestra', 'the salsoul orchestra', 'san remo golden strings', 'the san remo golden strings',
    'secrets', 'the secrets', 'shades of blue', 'the shades of blue',
    'shadows of knight', 'the shadows of knight', 'sherbs', 'the sherbs',
    'smashing pumpkins', 'the smashing pumpkins', 'sons of champlin', 'the sons of champlin',
    'soul sisters', 'the soul sisters', 'spinners', 'the spinners',
    'st. lunatics', 'the st. lunatics', 'status quo', 'the status quo',
    'streets', 'the streets', 'supremes', 'the supremes',
    'swans', 'the swans', 'sweet', 'the sweet',
    "swingin' medallions", "the swingin' medallions", 'temptations', 'the temptations',
    'undisputed truth', 'the undisputed truth', 'van dykes', 'the van dykes',
    'vanilla fudge', 'the vanilla fudge', 'whatnauts', 'the whatnauts',
}

kept = df_artists[df_artists['name'].isin(all_dup_names)][
    ['name', 'total_charting_songs']
].sort_values('name')
print(f'Kept {len(kept)} names (expected ~130):')
print(kept.to_string())



Kept 129 names (expected ~130):
                                   name  total_charting_songs
217                    addrisi brothers                     6
704              atlanta rhythm section                    14
847                   baja marimba band                     3
881                            bar-kays                     5
1070                    biddu orchestra                     2
1786                           bus boys                     1
1886                 captain & tennille                    13
1973                            casinos                     1
1992                         cathedrals                     0
2182                  chopper city boyz                     0
2361                  climax blues band                     4
2590                    crystal mansion                     1
3408                        dreamlovers                     1
3425                   dropkick murphys                     0
3448                             dukay

In [40]:
# Verify no remaining the/non-the duplicates slipped through, and find the missing pair
# Rebuilds name_no_the temporarily to check for any leftover collisions
def strip_leading_the(s):
    return re.sub(r'^the ', '', str(s)).strip()

df_artists['_tmp_no_the'] = df_artists['name'].apply(strip_leading_the)
remaining_dups = df_artists[df_artists.duplicated('_tmp_no_the', keep=False)]
print(f'Remaining the/non-the duplicates after dedup: {len(remaining_dups)}')
if len(remaining_dups):
    print(remaining_dups[['name', '_tmp_no_the', 'total_charting_songs']].sort_values('_tmp_no_the').to_string())

# Cross-check: find any pair from all_dup_names where NEITHER variant appears in df_artists
pair_keys = {}
for n in all_dup_names:
    key = strip_leading_the(n)
    pair_keys.setdefault(key, []).append(n)

in_df = set(df_artists['name'])
missing_pairs = []
for key, variants in pair_keys.items():
    if not any(v in in_df for v in variants):
        missing_pairs.append((key, variants))

print(f'\nPairs in all_dup_names with no surviving row in df_artists: {len(missing_pairs)}')
for key, variants in missing_pairs:
    print(f'  key={key!r}  variants={variants}')

df_artists = df_artists.drop(columns=['_tmp_no_the'])


Remaining the/non-the duplicates after dedup: 0

Pairs in all_dup_names with no surviving row in df_artists: 1
  key='5 stairsteps'  variants=['5 stairsteps', 'the 5 stairsteps']


In [42]:
# Drop known spelling-variant duplicates where the correct spelling is already present
TYPO_DROPS = {
    '30 secnds to mars',   # correct: '30 seconds to mars'
}

typo_mask = df_artists['name'].isin(TYPO_DROPS)
print(f'Dropping {typo_mask.sum()} typo variant(s):')
print(df_artists.loc[typo_mask, ['name', 'total_charting_songs']].to_string())
df_artists = df_artists[~typo_mask].copy()
print(f'df_artists: {df_artists.shape}')


Dropping 1 typo variant(s):
                 name  total_charting_songs
61  30 secnds to mars                     0
df_artists: (13610, 54)


In [43]:
# Drop names with '(' but no ')' — artifacts from comma-splitting
# e.g. "artist (qualifier, detail)" was split at the comma → "artist (qualifier" with no closing paren
# Intentionally truncated entries (already in PAREN_KEEP below) are excluded here
UNMATCHED_PAREN_KEEP = {
    '4*town (from disney',
    'mount westmore (snoop dogg',
    'silk sonic (bruno mars',
    'the swell season (glen hansard',
    'the barbusters (joan jett',
    'the ruffin brothers (jimmy',
    'i.a.p. co. (the italian asphalt',
}

unmatched_mask = (
    df_artists['name'].str.contains(r'\(', regex=True)
    & ~df_artists['name'].str.contains(r'\)', regex=True)
    & ~df_artists['name'].isin(UNMATCHED_PAREN_KEEP)
)

print(f'Dropping {unmatched_mask.sum()} unmatched-paren artifact(s):')
print(df_artists.loc[unmatched_mask, 'name'].to_string())
df_artists = df_artists[~unmatched_mask].copy()
print(f'df_artists: {df_artists.shape}')


Dropping 0 unmatched-paren artifact(s):
Series([], )
df_artists: (13610, 54)


In [46]:
# Load the saved df_artists_clean CSV and use it to restore the dropped rows.
# Update the path below to wherever your file is saved.
saved_path = base + 'df_artists_clean.csv'   # ← update if different

saved = pd.read_csv(saved_path)

# Keep only rows from the saved version that are missing from current df_artists
current_names = set(df_artists['name'])
missing = saved[~saved['name'].isin(current_names)]

# Align columns and concat
rows_to_add = missing[[c for c in df_artists.columns if c in missing.columns]].copy()
df_artists = (
    pd.concat([df_artists, rows_to_add], ignore_index=True)
    .sort_values('name')
    .reset_index(drop=True)
)
print(f'Reinstated {len(rows_to_add)} rows')
print(f'df_artists: {df_artists.shape}')


Reinstated 518 rows
df_artists: (13611, 54)


In [47]:
# Verify the reinstatement — check shape, spot-check known names, and flag any remaining issues

# 1. Shape
print(f'df_artists shape: {df_artists.shape}')

# 2. Spot-check a handful of names that should be back
spot_check = [
    'florence + the machine', 'kool & the gang', 'sly & the family stone',
    'tom petty & the heartbreakers', 'huey lewis & the news',
    'eric b. & rakim', 'dan + shay', 'she & him',
    'bruce springsteen & the e street band', 'echo & the bunnymen',
]
found = df_artists['name'].isin(spot_check)
print(f'\nSpot-check ({found.sum()}/{len(spot_check)} found):')
for name in spot_check:
    status = '✓' if name in df_artists['name'].values else '✗ MISSING'
    print(f'  {status}  {name}')

# 3. Confirm the two entries that SHOULD still be gone
should_be_gone = ['2pac & johnny p', '2pac + outlawz']
print(f'\nShould be absent:')
for name in should_be_gone:
    status = '✗ STILL PRESENT' if name in df_artists['name'].values else '✓ absent'
    print(f'  {status}  {name}')

# 4. Duplicate check
dupes = df_artists[df_artists.duplicated('name', keep=False)]
print(f'\nDuplicate names: {dupes["name"].nunique()} groups')
if len(dupes):
    print(dupes[['name']].drop_duplicates().to_string())


df_artists shape: (13611, 54)

Spot-check (10/10 found):
  ✓  florence + the machine
  ✓  kool & the gang
  ✓  sly & the family stone
  ✓  tom petty & the heartbreakers
  ✓  huey lewis & the news
  ✓  eric b. & rakim
  ✓  dan + shay
  ✓  she & him
  ✓  bruce springsteen & the e street band
  ✓  echo & the bunnymen

Should be absent:
  ✗ STILL PRESENT  2pac & johnny p
  ✗ STILL PRESENT  2pac + outlawz

Duplicate names: 0 groups


In [48]:
# Targeted drop of confirmed one-off collaboration credits
# Add any others you identify to this set
CONFIRMED_COLLAB_DROPS = {
    '2pac & johnny p',
    '2pac + outlawz',
}

drop_mask = df_artists['name'].isin(CONFIRMED_COLLAB_DROPS)
print(f'Dropping {drop_mask.sum()}:')
print(df_artists.loc[drop_mask, 'name'].tolist())
df_artists = df_artists[~drop_mask].copy()
print(f'df_artists: {df_artists.shape}')


Dropping 2:
['2pac & johnny p', '2pac + outlawz']
df_artists: (13609, 54)


In [7]:
# dropping additional unnecessary columns

# Drop specified columns from df_artists
cols_to_drop = [
    'first_album_year',
    'total_charting_songs',
    'total_charting_albums',
    '#1_hit_song_count',
    '#1_hit_album_count',
    'top_10_song_count',
    'top_10_album_count',
    'highest_charting_song_name',
    'highest_charting_song_position',
    'first_charting_song_name',
    'highest_charting_album_name',
    'highest_charting_album_position',
    'first_charting_album_name',
    'first_charting_album_position',
    'first_charting_album_duration',
    'first_year_top_10_songs',
    'first_year_num1_songs',
    'first_year_on_chart_albums',
    'first_year_top_10_albums',
    'first_year_num1_albums',
    'aggregate_labels',
    'first_top_10_song_#_of_connected_artists_to_labels_single_year',
    'first_top_10_song_#_of_connected_artists_to_labels_past_5_years',
    'first_top_10_song_total_power_of_connected_artists_to_labels_single_year',
    'first_top_10_song_total_power_of_connected_artists_to_labels_past_5_years',
    'first_top_10_song_#_of_connected_artists_to_big3_single_year',
    'first_top_10_song_#_of_connected_artists_to_big3_past_5_years',
    'first_top_10_song_total_power_of_connected_artists_to_big3_single_year',
    'first_top_10_song_total_power_of_connected_artists_to_big3_past_5_years',
    'first_top_10_hit_song_genre_tags',
    'first_top_10_hit_album_genre_tags',
    'top_10_1hit_wonder_song',
    'top_10_1hit_wonder_album',
    'top_10_hitmaker_songs',
    'top_10_hitmaker_albums',
]

df_artists = df_artists.drop(columns=cols_to_drop)
print(f"Remaining columns ({len(df_artists.columns)}): {df_artists.columns.tolist()}")


Remaining columns (19): ['name', 'musicbrainz_artist_id', 'musicbrainz_mbid', 'spotify_id', 'performer_pre_normalized', 'first_song_year', 'years_active_on_charts', 'first_charting_song_position', 'first_charting_song_duration', 'genre_tags_musicbrainz', 'first_year_on_chart_songs', 'genre_tags_through_first_top_10_hit', 'major_genre_categories_through_first_top_10_hit', '#_of_major_genre_categories_through_first_top_10_hit', 'musicbrainz_major_genre_categories', 'musicbrainz_#_of_genres', 'spotify_genres', 'spotify_major_genre_categories', 'combined_major_genre_categories']


# Start here

In [24]:
# Add 5 artist-level columns to df_artists derived from df_songs.
# Joins on musicbrainz_mbid (df_artists) ↔ musicbrainz_artist_id (df_songs),
# with artist name as fallback. Safe to re-run.

# --- Drop existing versions of new columns ---
all_new_cols = [
    'first_top_20_hit_year',
    'first_charting_song_year',
    'last_charting_song_year',
    'years_through_first_top_20_hit',
    '#_of_charting_songs_through_first_top_20_hit',
]
df_artists = df_artists.drop(columns=[c for c in all_new_cols if c in df_artists.columns])

agg_cols = [
    'first_top_20_hit_year',
    'first_charting_song_year',
    'last_charting_song_year',
    '#_of_charting_songs_through_first_top_20_hit',
]

# --- Compute aggregates from df_songs grouped by musicbrainz_artist_id (UUID) ---
top20 = df_songs[df_songs['peak_pos'] <= 20]

first_top20_yr = (
    top20.groupby('musicbrainz_artist_id', dropna=True)['first_charting_year']
    .min().rename('first_top_20_hit_year').reset_index()
)
charting_range = (
    df_songs.groupby('musicbrainz_artist_id', dropna=True)['first_charting_year']
    .agg(first_charting_song_year='min', last_charting_song_year='max')
    .reset_index()
)
aggs_by_id = charting_range.merge(first_top20_yr, on='musicbrainz_artist_id', how='left')
songs_aug = df_songs.merge(first_top20_yr, on='musicbrainz_artist_id', how='left')
counts_by_id = (
    songs_aug[songs_aug['first_charting_year'] <= songs_aug['first_top_20_hit_year']]
    .groupby('musicbrainz_artist_id').size()
    .rename('#_of_charting_songs_through_first_top_20_hit').reset_index()
)
aggs_by_id = aggs_by_id.merge(counts_by_id, on='musicbrainz_artist_id', how='left')

# --- Compute aggregates grouped by artist name (for fallback) ---
first_top20_yr_name = (
    top20.groupby('name', dropna=True)['first_charting_year']
    .min().rename('first_top_20_hit_year').reset_index()
)
charting_range_name = (
    df_songs.groupby('name', dropna=True)['first_charting_year']
    .agg(first_charting_song_year='min', last_charting_song_year='max')
    .reset_index()
)
aggs_by_name = charting_range_name.merge(first_top20_yr_name, on='name', how='left')
songs_aug_name = df_songs.merge(first_top20_yr_name, on='name', how='left')
counts_by_name = (
    songs_aug_name[songs_aug_name['first_charting_year'] <= songs_aug_name['first_top_20_hit_year']]
    .groupby('name').size()
    .rename('#_of_charting_songs_through_first_top_20_hit').reset_index()
)
aggs_by_name = aggs_by_name.merge(counts_by_name, on='name', how='left')

# --- Step 1: Merge by musicbrainz_mbid (df_artists) ↔ musicbrainz_artist_id (df_songs) ---
df_artists = df_artists.merge(
    aggs_by_id,
    left_on='musicbrainz_mbid',
    right_on='musicbrainz_artist_id',
    how='left'
).drop(columns=['musicbrainz_artist_id_y']).rename(columns={'musicbrainz_artist_id_x': 'musicbrainz_artist_id'})

# --- Step 2: Fill unmatched rows via name .map() ---
no_match_mask = df_artists['first_top_20_hit_year'].isna()
name_lookup = aggs_by_name.set_index('name')

for col in agg_cols:
    df_artists.loc[no_match_mask, col] = df_artists.loc[no_match_mask, 'name'].map(name_lookup[col])

# --- Step 3: Derived column ---
df_artists['years_through_first_top_20_hit'] = (
    df_artists['first_top_20_hit_year'] - df_artists['first_charting_song_year'] + 1
)

# --- Step 4: Reorder ---
existing = [c for c in df_artists.columns if c not in all_new_cols]
insert_at = existing.index('performer_pre_normalized') + 1
col_order = existing[:insert_at] + all_new_cols + existing[insert_at:]
df_artists = df_artists[col_order]

print(f"df_artists shape: {df_artists.shape}")
print(f"Matched by MBID: {(~no_match_mask).sum():,} | Matched by name: {df_artists.loc[no_match_mask, 'first_top_20_hit_year'].notna().sum():,} | No match: {df_artists['first_top_20_hit_year'].isna().sum():,}")
df_artists[['name', 'musicbrainz_mbid'] + all_new_cols].head(10)


df_artists shape: (13609, 24)
Matched by MBID: 2,853 | Matched by name: 396 | No match: 10,360


,name,musicbrainz_mbid,first_top_20_hit_year,first_charting_song_year,last_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit
0,!!! (chk chk chk),NaN,NaN,NaN,NaN,NaN,NaN
1,"""groove"" holmes",NaN,NaN,1966.0,1966.0,NaN,NaN
2,"""little"" jimmy dickens",NaN,1965.0,1965.0,1965.0,1.0,1.0
3,"""pookie"" hudson",29dc9009-015f-47c4-bd17-ed2af6d2ae0c,NaN,1963.0,1963.0,NaN,NaN
4,"""weird al"" yankovic",NaN,1984.0,1983.0,2014.0,2.0,4.0
5,$not,d993169f-2033-4810-bae8-564d4aab89cd,NaN,2021.0,2022.0,NaN,NaN
6,$uicideboy$,ef326b3d-61f0-4379-aca0-21237d696d63,NaN,2024.0,2026.0,NaN,NaN
7,'68,2bf3db78-1032-4111-9e1f-650b556bc2dd,NaN,NaN,NaN,NaN,NaN
8,'n sync,NaN,1998.0,1998.0,2002.0,1.0,3.0
9,'til tuesday,NaN,1985.0,1985.0,1989.0,1.0,2.0


In [25]:
# Cast year and count columns to nullable integer (supports NaN)
int_cols = [
    'first_top_20_hit_year',
    'first_charting_song_year',
    'last_charting_song_year',
    'years_through_first_top_20_hit',
    '#_of_charting_songs_through_first_top_20_hit',
]

for col in int_cols:
    df_artists[col] = df_artists[col].astype('Int64')

df_artists[int_cols].dtypes


first_top_20_hit_year                           Int64
first_charting_song_year                        Int64
last_charting_song_year                         Int64
years_through_first_top_20_hit                  Int64
#_of_charting_songs_through_first_top_20_hit    Int64
dtype: object

In [26]:
df_artists.head()

,name,musicbrainz_artist_id,musicbrainz_mbid,spotify_id,performer_pre_normalized,first_top_20_hit_year,first_charting_song_year,last_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,first_song_year,years_active_on_charts,first_charting_song_position,first_charting_song_duration,genre_tags_musicbrainz,first_year_on_chart_songs,genre_tags_through_first_top_10_hit,major_genre_categories_through_first_top_10_hit,#_of_major_genre_categories_through_first_top_10_hit,musicbrainz_major_genre_categories,musicbrainz_#_of_genres,spotify_genres,spotify_major_genre_categories,combined_major_genre_categories
0,!!! (chk chk chk),<NA>,NaN,NaN,!!! (Chk Chk Chk),<NA>,<NA>,<NA>,<NA>,<NA>,NaN,2007-2007,0,0,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN
1,"""groove"" holmes",<NA>,NaN,NaN,"""Groove"" Holmes",<NA>,1966,1966,<NA>,<NA>,1966.0,1966-1966,44,11,NaN,1966.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN
2,"""little"" jimmy dickens",<NA>,NaN,NaN,"""Little"" Jimmy Dickens",1965,1965,1965,1,1,1965.0,1965-1965,15,10,NaN,1965.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN
3,"""pookie"" hudson",232958.0,29dc9009-015f-47c4-bd17-ed2af6d2ae0c,NaN,"""Pookie"" Hudson",<NA>,1963,1963,<NA>,<NA>,1963.0,1963-1963,96,1,NaN,1963.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN
4,"""weird al"" yankovic",<NA>,NaN,NaN,"""Weird Al"" Yankovic",1984,1983,2014,2,4,1983.0,1983-2014,63,8,NaN,1983.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN


In [29]:
# Merge network metrics into df_artists

# Load df_artists_network_metrics from CSV.

import pandas as pd

df_artists_network_metrics = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists_network_metrics.csv'
)

print(df_artists_network_metrics.shape)
df_artists_network_metrics.head()


(13609, 26)


,name,musicbrainz_artist_id,degree_centrality_top20_yearly,degree_centrality_firstsong_yearly,degree_centrality_top20_rolling5,degree_centrality_firstsong_rolling5,closeness_centrality_top20_yearly,closeness_centrality_firstsong_yearly,closeness_centrality_top20_rolling5,closeness_centrality_firstsong_rolling5,harmonic_closeness_centrality_top20_yearly,harmonic_closeness_centrality_firstsong_yearly,harmonic_closeness_centrality_top20_rolling5,harmonic_closeness_centrality_firstsong_rolling5,betweenness_centrality_top20_yearly,betweenness_centrality_firstsong_yearly,betweenness_centrality_top20_rolling5,betweenness_centrality_firstsong_rolling5,eigenvector_centrality_top20_yearly,eigenvector_centrality_firstsong_yearly,eigenvector_centrality_top20_rolling5,eigenvector_centrality_firstsong_rolling5,power_of_connected_artists_top20_yearly,power_of_connected_artists_firstsong_yearly,power_of_connected_artists_top20_rolling5,power_of_connected_artists_firstsong_rolling5
0,!!! (chk chk chk),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"""groove"" holmes",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"""little"" jimmy dickens",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"""pookie"" hudson",232958.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"""weird al"" yankovic",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Merge 5 network metric columns into df_artists.
# Drops NaN musicbrainz_artist_id rows from metrics before merging
# to prevent NaN-to-NaN cartesian product explosion.

metric_cols = [
    'degree_centrality_top20_rolling5',
    'harmonic_closeness_centrality_top20_rolling5',
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
    'power_of_connected_artists_top20_rolling5',
]

df_artists = df_artists.drop(columns=[c for c in metric_cols if c in df_artists.columns])

metrics_to_merge = df_artists_network_metrics[['musicbrainz_artist_id'] + metric_cols].copy()
for col in ['musicbrainz_artist_id'] + metric_cols:
    metrics_to_merge[col] = pd.to_numeric(metrics_to_merge[col], errors='coerce')

metrics_to_merge = metrics_to_merge.dropna(subset=['musicbrainz_artist_id'])

df_artists['musicbrainz_artist_id'] = pd.to_numeric(df_artists['musicbrainz_artist_id'], errors='coerce')

df_artists = df_artists.merge(metrics_to_merge, on='musicbrainz_artist_id', how='left')

print(df_artists.shape)
print(df_artists[metric_cols].notna().sum())


(13655, 29)
degree_centrality_top20_rolling5                1322
harmonic_closeness_centrality_top20_rolling5    1322
betweenness_centrality_top20_rolling5           1322
eigenvector_centrality_top20_rolling5           1322
power_of_connected_artists_top20_rolling5       1076
dtype: int64


In [41]:
# For each artist's first top-20 hit song in df_songs, map aggregate_major_genre_category_musicbrainz
# to df_artists as 'first_top_20_song_major_genres', placed after combined_major_genre_categories.
# Primary join: df_songs['musicbrainz_artist_id'] (UUID) ↔ df_artists['musicbrainz_mbid']
# Fallback join: name ↔ name, for artists with no UUID match.

# Step 1: isolate first top-20 hit per artist in df_songs
top20_songs = (
    df_songs[df_songs['peak_pos'] <= 20]
    .sort_values(['musicbrainz_artist_id', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['musicbrainz_artist_id'], keep='first')
)[['musicbrainz_artist_id', 'name', 'aggregate_major_genre_category_musicbrainz']]

# Step 2: build UUID-keyed and name-keyed lookups
uuid_lookup = (
    top20_songs.dropna(subset=['musicbrainz_artist_id'])
    .set_index('musicbrainz_artist_id')['aggregate_major_genre_category_musicbrainz']
)
name_lookup = (
    top20_songs.dropna(subset=['name'])
    .drop_duplicates(subset=['name'], keep='first')
    .set_index('name')['aggregate_major_genre_category_musicbrainz']
)

# Step 3: map via musicbrainz_mbid first, then name fallback
df_artists['first_top_20_song_major_genres'] = df_artists['musicbrainz_mbid'].map(uuid_lookup)

no_match = df_artists['first_top_20_song_major_genres'].isna()
df_artists.loc[no_match, 'first_top_20_song_major_genres'] = df_artists.loc[no_match, 'name'].map(name_lookup)

# Step 4: move column to after combined_major_genre_categories
ref_col = 'combined_major_genre_categories'
cols = list(df_artists.columns)
cols.remove('first_top_20_song_major_genres')
insert_at = cols.index(ref_col) + 1
cols.insert(insert_at, 'first_top_20_song_major_genres')
df_artists = df_artists[cols]

print(df_artists['first_top_20_song_major_genres'].notna().sum(), 'artists mapped')
print(df_artists[['name', 'first_top_20_song_major_genres']].dropna().head(10).to_string(index=False))


1281 artists mapped
          name                                                   first_top_20_song_major_genres
        *nsync                                                                              Pop
10,000 maniacs                                                                             Rock
           112              Electronic/Dance, Hip Hop/Rap, Pop, R&B/Soul/Funk, Reggae/Caribbean
    20 fingers                                                                 Electronic/Dance
      24kgoldn Electronic/Dance, Hip Hop/Rap, Latin, Pop, R&B/Soul/Funk, Reggae/Caribbean, Rock
  3 doors down                                                               Blues, Metal, Rock
    38 special                                                                        Pop, Rock
            3t                                                       Hip Hop/Rap, R&B/Soul/Funk
 4 non blondes                                                                        Pop, Rock
      40 thevz      

In [54]:
# Calculate average weeks on chart for top-20 hit songs with first_charting_year 2000–2020.

mask = (
    (df_songs['peak_pos'] <= 20) &
    (df_songs['first_charting_year'].between(2000, 2020))
)

avg_wks = df_songs[mask]['wks_on_chart'].mean()
print(f"Average weeks on chart: {avg_wks:.1f}")

# Calculate deciles of weeks on chart for top-20 songs with first_charting_year 2000–2020.

import numpy as np

deciles = df_songs[mask]['wks_on_chart'].quantile(np.arange(0.1, 1.1, 0.1))
deciles.index = [f"{int(q*100)}th" for q in deciles.index]
print(deciles.to_string())



Average weeks on chart: 24.8
10th     11.0
20th     20.0
30th     20.0
40th     21.0
50th     23.0
60th     26.0
70th     28.0
80th     32.0
90th     39.0
100th    90.0


In [57]:
# Add weeks on chart metric but only for top 20 hit songs

# Add wks_on_chart from df_songs to df_artists for each artist's first top-20 hit song.
# Primary join: df_songs['musicbrainz_artist_id'] (UUID) ↔ df_artists['musicbrainz_mbid']
# Fallback join: name ↔ name

# Step 1: isolate first top-20 hit per artist
first_top20 = (
    df_songs[df_songs['peak_pos'] <= 20]
    .sort_values(['musicbrainz_artist_id', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['musicbrainz_artist_id'], keep='first')
)[['musicbrainz_artist_id', 'name', 'wks_on_chart']]

# Step 2: build lookups
uuid_lookup = (
    first_top20.dropna(subset=['musicbrainz_artist_id'])
    .set_index('musicbrainz_artist_id')['wks_on_chart']
)
name_lookup = (
    first_top20.dropna(subset=['name'])
    .drop_duplicates(subset=['name'], keep='first')
    .set_index('name')['wks_on_chart']
)

# Step 3: map to df_artists
df_artists['top_20_hit_song_#_wks_on_chart_any_position'] = df_artists['musicbrainz_mbid'].map(uuid_lookup)

no_match = df_artists['top_20_hit_song_#_wks_on_chart_any_position'].isna()
df_artists.loc[no_match, 'top_20_hit_song_#_wks_on_chart_any_position'] = \
    df_artists.loc[no_match, 'name'].map(name_lookup)

print(df_artists['top_20_hit_song_#_wks_on_chart_any_position'].notna().sum(), 'artists mapped')


2890 artists mapped


In [ ]:
# Move top_20_hit_song_#_wks_on_chart_any_position to after first_charting_song_duration.

cols = list(df_artists.columns)
cols.remove('top_20_hit_song_#_wks_on_chart_any_position')
insert_at = cols.index('first_charting_song_duration') + 1
cols.insert(insert_at, 'top_20_hit_song_#_wks_on_chart_any_position')
df_artists = df_artists[cols]


,name,musicbrainz_artist_id,musicbrainz_mbid,spotify_id,performer_pre_normalized,first_top_20_hit_year,first_charting_song_year,last_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,first_song_year,years_active_on_charts,first_charting_song_position,first_charting_song_duration,top_20_hit_song_#_wks_on_chart_any_position,genre_tags_musicbrainz,first_year_on_chart_songs,genre_tags_through_first_top_10_hit,major_genre_categories_through_first_top_10_hit,#_of_major_genre_categories_through_first_top_10_hit,musicbrainz_major_genre_categories,musicbrainz_#_of_genres,spotify_genres,spotify_major_genre_categories,combined_major_genre_categories,first_top_20_song_major_genres,degree_centrality_top20_rolling5,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,power_of_connected_artists_top20_rolling5
0,!!! (chk chk chk),NaN,NaN,NaN,!!! (Chk Chk Chk),NaN,NaN,NaN,NaN,NaN,NaN,2007-2007,0,0,NaN,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"""groove"" holmes",NaN,NaN,NaN,"""Groove"" Holmes",NaN,1966.0,1966.0,NaN,NaN,1966.0,1966-1966,44,11,NaN,NaN,1966.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Load google_trends_top3000.csv as google_trends_top3000.

google_trends_top3000 = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/google_trends_top3000.csv'
)

print(google_trends_top3000.shape)
google_trends_top3000.head()

# Load billboard_hot100_1958_2026.csv as billboard_hot100_songs, replacing the existing dataframe.

billboard_hot100_songs = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/billboard_hot100_1958_2026.csv'
)

print(billboard_hot100_songs.shape)
billboard_hot100_songs.head()



(215, 5555)


,date,"""Weird Al"" Yankovic_web","""Weird Al"" Yankovic_youtube",'N Sync_web,'Til Tuesday_web,'Til Tuesday_youtube,"10,000 Maniacs_web","10,000 Maniacs_youtube",100 Proof Aged in Soul_web,100 Proof Aged in Soul_youtube,10cc_web,10cc_youtube,112_web,112_youtube,1910 Fruitgum Co._web,2 Chainz Featuring Drake_web,2 Chainz Featuring Drake_youtube,2 Chainz_web,2 Chainz_youtube,2 In A Room_web,2 In A Room_youtube,2 Unlimited_web,2 Unlimited_youtube,20 Fingers Featuring Gillette_web,21 Savage & Metro Boomin_web,21 Savage & Metro Boomin_youtube,21 Savage_web,21 Savage_youtube,24kGoldn Featuring iann dior_web,24kGoldn Featuring iann dior_youtube,2Pac_web,2Pac_youtube,3 Doors Down_web,3 Doors Down_youtube,30 Seconds To Mars_web,30 Seconds To Mars_youtube,38 Special_web,38 Special_youtube,3LW_web,3LW_youtube,3OH!3_web,3OH!3_youtube,3T_web,3T_youtube,3rd Party_web,3rd Party_youtube,4 Non Blondes_web,4 Non Blondes_youtube,4PM_web,4PM_youtube,5 Seconds Of Summer_web,5 Seconds Of Summer_youtube,50 Cent_web,50 Cent_youtube,69 Boyz_web,69 Boyz_youtube,6ix9ine_web,6ix9ine_youtube,702_web,702_youtube,95 South_web,95 South_youtube,98 Degrees_web,98 Degrees_youtube,A Boogie Wit da Hoodie_web,A Boogie Wit da Hoodie_youtube,A Flock Of Seagulls_web,A Flock Of Seagulls_youtube,A Great Big World & Christina Aguilera_web,A Great Big World & Christina Aguilera_youtube,A Taste Of Honey_web,A Taste Of Honey_youtube,A Tribe Called Quest_web,A Tribe Called Quest_youtube,"A$AP Rocky Featuring Drake, 2 Chainz & Kendrick Lamar_web",A'me Lorain_web,A'me Lorain_youtube,AB Logic_web,AB Logic_youtube,ABBA_web,ABBA_youtube,ABC_web,ABC_youtube,AC/DC_web,AC/DC_youtube,AFI_web,AFI_youtube,AJR_web,AJR_youtube,AWB_web,AWB_youtube,AWOLNATION_web,AWOLNATION_youtube,Aaliyah_web,Aaliyah_youtube,Aaron Hall_web,Aaron Hall_youtube,Aaron Neville_web,Aaron Neville_youtube,Aaron Tippin_web,Aaron Tippin_youtube,Ace Cannon_web,Ace Cannon_youtube,Ace Of Base_web,Ace Of Base_youtube,Adam Ant_web,Adam Ant_youtube,Adam Lambert_web,Adam Lambert_youtube,Adam Wade_web,Adam Wade_youtube,Addrisi Brothers_web,Addrisi Brothers_youtube,Adele_web,Adele_youtube,Adina Howard_web,Adina Howard_youtube,Aerosmith_web,Aerosmith_youtube,After 7_web,After 7_youtube,After The Fire_web,After The Fire_youtube,Air Supply_web,Air Supply_youtube,Akon Featuring Eminem_web,Akon Featuring Eminem_youtube,Akon Featuring Snoop Dogg_web,Akon Featuring Snoop Dogg_youtube,Akon Featuring Styles P._youtube,Akon_web,Akon_youtube,Al B. Sure!_web,Al B. Sure!_youtube,Al Green_web,Al Green_youtube,Al Jarreau_web,Al Jarreau_youtube,Al Martino_web,Al Martino_youtube,Al Stewart_web,Al Stewart_youtube,Al Wilson_web,Al Wilson_youtube,Alabama_web,Alabama_youtube,Alan Jackson_web,Alan Jackson_youtube,Alan O'Day_web,Alan O'Day_youtube,Alanis Morissette_web,Alanis Morissette_youtube,Alannah Myles_web,Alannah Myles_youtube,Albert Hammond_web,Albert Hammond_youtube,Alessia Cara_web,Alessia Cara_youtube,Alex Clare_web,Alex Clare_youtube,Alexander O'Neal_web,Alexander O'Neal_youtube,Alias_web,Alias_youtube,Alice Cooper_web,Alice Cooper_youtube,Alicia Bridges_web,Alicia Bridges_youtube,Alicia Keys Featuring Nicki Minaj_web,Alicia Keys Featuring Nicki Minaj_youtube,Alicia Keys_web,Alicia Keys_youtube,Alisha_web,Alisha_youtube,All Saints_web,All Saints_youtube,All-4-One_web,All-4-One_youtube,Allan Sherman_web,Allan Sherman_youtube,Allure Featuring 112_youtube,Alphaville_web,Alphaville_youtube,Aly & AJ_web,Aly & AJ_youtube,Amanda Perez_web,Amanda Perez_youtube,Amazing Rhythm Aces_web,Amazing Rhythm Aces_youtube,Amber_web,Amber_youtube,Ambrosia_web,Ambrosia_youtube,America_web,America_youtube,American Authors_web,American Authors_youtube,Amerie_web,Amerie_youtube,Amii Stewart_web,Amii Stewart_youtube,Amine_web,Amine_youtube,Amy Grant_web,Amy Grant_youtube,Amy Winehouse_web,Amy Winehouse_youtube,Andrea True Connection_web,Andrea True Connection_youtube,Andrew Gold_web,Andrew Gold_youtube,Andy Gibb_web,Andy Gibb_youtube,Andy Grammer_web,Andy Gr

In [69]:
# Find the first charting month and year per song from the week-by-week Billboard data,
# then add it to df_songs as first_charting_month_and_year after first_charting_year.
# Joins on title + performer (lowercased) to handle minor capitalization differences.

import pandas as pd

# Step 1: find first chart_week per title+performer
billboard_hot100_songs['chart_week'] = pd.to_datetime(billboard_hot100_songs['chart_week'])

first_chart_date = (
    billboard_hot100_songs
    .assign(
        _title    = billboard_hot100_songs['title'].str.lower().str.strip(),
        _performer = billboard_hot100_songs['performer'].str.lower().str.strip()
    )
    .groupby(['_title', '_performer'])['chart_week']
    .min()
    .reset_index()
)
first_chart_date['first_charting_month_and_year'] = (
    first_chart_date['chart_week'].dt.strftime('%B %Y')
)

# Step 2: map to df_songs via title + name (both lowercased)
df_songs['_title']     = df_songs['title'].str.lower().str.strip()
df_songs['_performer'] = df_songs['name'].str.lower().str.strip()

df_songs = df_songs.merge(
    first_chart_date[['_title', '_performer', 'first_charting_month_and_year']],
    on=['_title', '_performer'],
    how='left'
).drop(columns=['_title', '_performer'])

# Step 3: move column to after first_charting_year
cols = list(df_songs.columns)
cols.remove('first_charting_month_and_year')
cols.insert(cols.index('first_charting_year') + 1, 'first_charting_month_and_year')
df_songs = df_songs[cols]

print(df_songs['first_charting_month_and_year'].notna().sum(), '/', len(df_songs), 'songs mapped')
print(df_songs[['title', 'name', 'first_charting_year', 'first_charting_month_and_year']].head(5).to_string(index=False))


27350 / 38383 songs mapped
                          title             name  first_charting_year first_charting_month_and_year
           It's All In The Game    tommy edwards                 1958                   August 1958
         It's Only Make Believe    conway twitty                 1958                September 1958
                    Little Star     the elegants                 1958                   August 1958
Nel Blu Dipinto Di Blu (Volaré) domenico modugno                 1958                   August 1958
               Poor Little Fool     ricky nelson                 1958                   August 1958


In [84]:
# Merge Spotify audio feature columns from each artist's first top-20 hit song
# into df_artists, prefixed with 'first_top_20_song_', placed after first_top_20_song_major_genres.
# Primary join: df_songs['musicbrainz_artist_id'] (UUID) ↔ df_artists['musicbrainz_mbid']
# Fallback: name ↔ name

audio_cols = ['duration_ms', 'acousticness', 'danceability', 'energy',
              'instrumentalness', 'liveness', 'loudness', 'speechiness',
              'tempo', 'valence', 'mode', 'explicit']

new_cols = [f'first_top_20_song_{c}' for c in audio_cols]

# Drop any existing versions
df_artists = df_artists.drop(columns=[c for c in new_cols if c in df_artists.columns])

# Step 1: isolate first top-20 hit per artist
first_top20_audio = (
    df_songs[df_songs['peak_pos'] <= 20]
    .sort_values(['musicbrainz_artist_id', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['musicbrainz_artist_id'], keep='first')
)[['musicbrainz_artist_id', 'name'] + audio_cols]

# Rename audio cols with prefix
first_top20_audio = first_top20_audio.rename(
    columns={c: f'first_top_20_song_{c}' for c in audio_cols}
)

# Step 2: UUID-keyed lookup
uuid_lookup = (
    first_top20_audio.dropna(subset=['musicbrainz_artist_id'])
    .set_index('musicbrainz_artist_id')[new_cols]
)

# Step 3: name-keyed lookup (for fallback)
name_lookup = (
    first_top20_audio.dropna(subset=['name'])
    .drop_duplicates(subset=['name'], keep='first')
    .set_index('name')[new_cols]
)

# Step 4: map via musicbrainz_mbid first, then name fallback
for col in new_cols:
    df_artists[col] = df_artists['musicbrainz_mbid'].map(uuid_lookup[col])

no_match = df_artists[new_cols[0]].isna()
for col in new_cols:
    df_artists.loc[no_match, col] = df_artists.loc[no_match, 'name'].map(name_lookup[col])

# Step 5: reorder — insert all new cols after first_top_20_song_major_genres
cols = list(df_artists.columns)
for c in new_cols:
    cols.remove(c)
insert_at = cols.index('first_top_20_song_major_genres') + 1
for i, c in enumerate(new_cols):
    cols.insert(insert_at + i, c)
df_artists = df_artists[cols]

print(df_artists[new_cols].notna().sum())


first_top_20_song_duration_ms         1628
first_top_20_song_acousticness        1628
first_top_20_song_danceability        1628
first_top_20_song_energy              1628
first_top_20_song_instrumentalness    1628
first_top_20_song_liveness            1628
first_top_20_song_loudness            1628
first_top_20_song_speechiness         1628
first_top_20_song_tempo               1628
first_top_20_song_valence             1628
first_top_20_song_mode                1628
first_top_20_song_explicit            1628
dtype: int64


In [88]:
# Add binary target column 'top_20_hitmaker' to df_artists.
# 1 if the artist had more than 1 top-20 song, 0 if exactly 1, NaN if no top-20 songs.

top20_counts = (
    df_songs[df_songs['peak_pos'] <= 20]
    .groupby('name')
    .size()
    .reset_index(name='_n_top20')
)

df_artists = df_artists.merge(top20_counts, on='name', how='left')
df_artists['top_20_hitmaker'] = df_artists['_n_top20'].apply(
    lambda x: 1 if x > 1 else (0 if x == 1 else None)
)
df_artists = df_artists.drop(columns=['_n_top20'])

print(df_artists['top_20_hitmaker'].value_counts(dropna=False))


top_20_hitmaker
NaN    10396
0.0     1867
1.0     1392
Name: count, dtype: int64


In [90]:
# Count top_20_hitmaker values (1 vs 0) among artists whose first top-20 hit was 2000–2020.

mask = df_artists['first_top_20_hit_year'].between(2000, 2020)
counts = df_artists[mask]['top_20_hitmaker'].value_counts(dropna=False)
total = mask.sum()

for val, n in counts.items():
    label = 'Hitmaker (>1 top-20)' if val == 1 else ('One-hit wonder (1 top-20)' if val == 0 else 'No top-20')
    print(f"{label}: {n} ({100 * n / total:.1f}%)")


One-hit wonder (1 top-20): 454 (56.0%)
Hitmaker (>1 top-20): 347 (42.8%)
No top-20: 10 (1.2%)


In [ ]:
# Rename musicbrainz_artist_id to musicbrainz_artist_mbid in df_songs and save to CSV
df_songs = df_songs.rename(columns={'musicbrainz_artist_id': 'musicbrainz_artist_mbid'})
df_songs.to_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv', index=False)
print(f"Saved {len(df_songs):,} rows")


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid,label,major_label,major_label_names,other_label,other_label_names,warner,universal,sony,label_mbid,song_genre_tags_musicbrainz,song_tags_musicbrainz,album_genre_tags_musicbrainz,song_major_genre_category_musicbrainz,album_major_genre_category_musicbrainz,aggregate_major_genre_category_musicbrainz,#_of_genres_song,#_of_genres_aggregate,spotify_id,duration_ms,release_date,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,mode,key,popularity,explicit
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958,0b1a0ca5-52cb-4d1d-8344-746f905ae115,7cf06b8f-a898-4f2c-a589-a98af8af6001,['Not Now Music'],False,NaN,True,['Not Now Music'],False,False,False,['b454b444-eb75-46ed-b400-19e85d8e1c64'],NaN,NaN,NaN,NaN,NaN,NaN,0,0,2lmPUdIdzlFH64PWJrw6Zb,159040.0,1/1/94,0.038,0.384,0.276,0.000000,0.309,-13.527,0.0295,105.181,0.654,1.0,3.0,49.0,0.0
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958,a3c60d26-90d6-4788-ba9b-a89693fc396d,4123e40f-d667-4634-919d-6e1d77a19934,['Remasters Workshop'],False,NaN,True,['Remasters Workshop'],False,False,False,['e55f5360-a4d0-43a1-b84d-f69b8510f3c6'],NaN,NaN,NaN,NaN,NaN,NaN,0,0,1xVOttVNT27FBTD8iHjOfU,132027.0,1/1/59,0.860,0.461,0.466,0.000028,0.135,-9.627,0.0598,128.537,0.251,1.0,11.0,51.0,0.0
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958,91919ad2-80cb-4077-bbb3-2803fa129fab,e15d4722-47a7-4874-99f6-285529503569,['Club Records'],False,NaN,True,['Club Records'],False,False,False,['9019dc41-f533-4c25-81b1-35fc98531903'],NaN,NaN,NaN,NaN,NaN,NaN,0,0,3c7KT5CN8uYRaK3xThhdYt,162773.0,7/26/05,0.873,0.408,0.397,0.000000,0.280,-12.536,0.0300,72.615,0.697,1.0,9.0,40.0,0.0
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958,b8e60837-e646-4f18-8f92-27d1173511b0,858ea2b4-7cc8-4a13-aa93-a3a5a73e71fd,['Warner Music Italy'],True,['Warner Music Italy'],False,NaN,True,False,False,['30d5acae-ae46-4689-9a91-99293a55e95e'],NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958,28d0c272-4d51-4c24-b31f-e20aac2ba7de,100b7f30-1519-427a-adc2-2b0d6a737db1,['Chartbuster Karaoke'],False,NaN,True,['Chartbuster Karaoke'],False,False,False,['4c44731f-f69b-4bac-b396-064980e894eb'],NaN,NaN,NaN,NaN,NaN,NaN,0,0,1ugZWl7RmEq95dea9hqorZ,153667.0,1/1/90,0.702,0.522,0.227,0.000095,0.238,-16.426,0.0318,155.514,0.839,1.0,0.0,35.0,0.0


In [91]:
# Save cleaned df_artists to CSV
out_path = '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv'
df_artists.to_csv(out_path, index=False)
print(f'Saved df_artists: {df_artists.shape}')
print(f'Path: {out_path}')


Saved df_artists: (13655, 44)
Path: /Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv
